In [1]:
import pandas as pd
import pickle

from datetime import datetime

In [2]:
test = pd.read_csv("test.csv")

test_bkup = test.drop(columns=["client", "close", "price_am", "price_pm"])

In [3]:
# datetime列をdatetime64型に変換
test["datetime"] = pd.to_datetime(test["datetime"])

# year, month, day, weekday列を作成
test["year"] = test["datetime"].dt.year
test["month"] = test["datetime"].dt.month
test["day"] = test["datetime"].dt.day
test["weekday"] = test["datetime"].dt.dayofweek

# 不要なdatetime列を削除
test.drop("datetime", axis=1, inplace=True)

test.head()

,client,close,price_am,price_pm,year,month,day,weekday
0,1,0,3,2,2016,4,1,4
1,0,0,5,5,2016,4,2,5
2,1,0,2,2,2016,4,3,6
3,1,0,1,1,2016,4,4,0
4,0,0,1,1,2016,4,5,1


In [4]:
#休日フラグがオンなら無条件で0にする
test.loc[test["close"] == 1, ["price_am", "price_pm"]] = 0

In [5]:
#clientが1とそれ以外に分割
client_test = test.query("client == 1")
test = test.query("client == 0")

In [6]:
#モデルの読み込み
with open("Apple_client_model.pickle", "rb") as fp:
    client_model = pickle.load(fp)
    
with open("Apple_model.pickle", "rb") as fp:
    model = pickle.load(fp)

In [7]:
#予測
client_y_pred = client_model.predict(client_test)

y_pred = model.predict(test)

NotFittedError: need to call fit or load_model beforehand

In [ ]:
client_test.loc[:, "y"] = client_y_pred

test.loc[:, "y"] = y_pred

In [ ]:
#休日フラグがオンなら無条件で0にする
client_test.loc[client_test["close"] == 1, ["y"]] = 0
test.loc[test["close"] == 1, ["y"]] = 0

In [ ]:
inference_client = client_test.drop(columns=["client", "close", "price_am", "price_pm", "year", "month", "day",  "weekday"])

inference = test.drop(columns=["client", "close", "price_am", "price_pm", "year", "month", "day",  "weekday"])

In [ ]:
inference_client.head()

In [ ]:
inference.head()

In [ ]:
inference = pd.concat([test_bkup, inference_client, inference])

inference = inference.sort_index()

inference = inference.groupby(inference.index).first()

inference.head()

In [ ]:
inference.to_csv("inference.csv", index=False, header=None)